# Score notes as a bopp extension

Brian McFee's PR #8 (bmcfee/bopp) adds `ext` payloads, whose items are validated in a second pass by a type that an installed package registers under the `bopp_extension` entry-point group. The `bopp-score-ext` plugin in `experiments/plugin/` registers `io.github.johentsch.score_note:v1` as a `ScoreNote` struct that mirrors the columnar `score_note` payload. This notebook builds a `score_note` annotation from a Distant Listening Corpus note table, re-encodes it as an `ext` annotation, and shows the ids, file sizes, the no-plugin fallback, validation errors and timings.

**Setup**
- Needs the plugin installed (`pip install -e experiments/plugin`), otherwise the schema is not in the registry.
- Reads 822 notes from a DLC fixture into a columnar `score_note` annotation, the baseline to compare against.

In [1]:
import tempfile
import timeit
import warnings
from pathlib import Path

import msgspec
import pandas as pd
from bopp_score_ext import EXT_SCHEMA, from_ext, to_ext

import bopp
import bopp.core
import bopp.extensions
import bopp.util
from bopp.io import load_bopp_msgpack, save_bopp_json, save_bopp_msgpack
from experiments.dlc_notes import notes_to_score_note, read_notes_tsv

# bopp is imported from the repository checkout, so the fixture is located from the package,
# not from the kernel working directory (the notebook may be run from a copy elsewhere)
REPO = Path(bopp.__file__).resolve().parents[1]
FIXTURE = REPO / "tests" / "experiments" / "data" / "dlc_notes_chopin_mazurkas_BI163op67-4.tsv"
WORK = Path(tempfile.mkdtemp())

ann = notes_to_score_note(read_notes_tsv(FIXTURE))
bopp.util.to_dataframe(ann).head(3)

,payload:score_note:midi,payload:score_note:tpc,payload:score_note:name,payload:score_note:octave,payload:score_note:staff,payload:score_note:voice,payload:score_note:mc,payload:score_note:mn,payload:score_note:tied,payload:score_note:gracenote,payload:score_note:nominal_duration,payload:score_note:scalar,payload:score_note:chord_id,payload:score_note:volta,extent:quarters_interval.fraction:quarter,extent:quarters_interval.fraction:duration
0,76,4,E5,5,1,1,1,0,1.0,None,"[1, 4]","[1, 1]",0,NaN,"[0, 1]","[1, 1]"
1,45,3,A2,2,2,1,2,1,NaN,None,"[1, 4]","[1, 1]",5,NaN,"[1, 1]","[1, 1]"
2,76,4,E5,5,1,1,2,1,-1.0,None,"[1, 4]","[3, 2]",1,NaN,"[1, 1]","[3, 2]"


**Columnar → `ext`**
- `to_ext` turns the columns into one `ScoreNote` row per note under the plugin's `ext_schema`.
- The two forms are different payloads, so they get different ids.

In [2]:
ext = to_ext(ann)
{
    "ext_schema": ext.payload.ext_schema,
    "first row": ext.payload.value[0],
    "columnar id": ann.id,
    "ext id": ext.id,
}

{'ext_schema': 'io.github.johentsch.score_note:v1',
 'first row': ScoreNote(midi=76, tpc=4, name='E5', octave=5, staff=1, voice=1, mc=1, mn=0, measure=UNSET, tied=1, gracenote=UNSET, nominal_duration=FractionPair(numerator=1, denominator=4), scalar=FractionPair(numerator=1, denominator=1), chord_id=0, tremolo=UNSET, volta=UNSET, tuning=UNSET, anchor=UNSET),
 'columnar id': '9e09e71c-b72a-5ea7-af07-430412d1d877',
 'ext id': '0708a693-7d74-5267-abe0-7c04fa7ea9d9'}

**File sizes**
- Saves both forms as JSON and MsgPack. The row form repeats the field names in every row, so its files are larger.

In [3]:
save_bopp_json(ann, WORK / "columnar.json")
save_bopp_msgpack(ann, WORK / "columnar.msgpack")
save_bopp_json(ext, WORK / "ext.json")
save_bopp_msgpack(ext, WORK / "ext.msgpack")
pd.DataFrame(
    {fmt: {form: (WORK / f"{form}.{fmt}").stat().st_size for form in ("columnar", "ext")} for fmt in ("json", "msgpack")}
).rename_axis(index="payload", columns="bytes")

bytes,json,msgpack
payload,,
columnar,54919,23490
ext,122046,82927


**Ids with and without resolution**
- Loads the `ext` file with rows resolved to `ScoreNote` and left as dicts.
- Both give the same, valid id, so the id doesn't depend on whether the plugin is present.

In [4]:
resolved = load_bopp_msgpack(WORK / "ext.msgpack")
unresolved = load_bopp_msgpack(WORK / "ext.msgpack", resolve_ext=False)
{
    "first row, resolved": type(resolved.payload.value[0]).__name__,
    "first row, unresolved": type(unresolved.payload.value[0]).__name__,
    "same id": ext.id == resolved.id == unresolved.id,
    "ids valid": bopp.core.validate_annotation_id(resolved) and bopp.core.validate_annotation_id(unresolved),
}

{'first row, resolved': 'ScoreNote',
 'first row, unresolved': 'dict',
 'same id': True,
 'ids valid': True}

**Plugin missing**
- Empties the registry to simulate a reader without the plugin.
- Loading warns and keeps the rows as dicts. The file still loads and its id is unchanged.

In [5]:
bopp.extensions.REGISTRY.clear()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    without_plugin = load_bopp_msgpack(WORK / "ext.msgpack")
bopp.extensions.reset_extensions()
{
    "warning": next(str(w.message) for w in caught if "No extension registered" in str(w.message)),
    "row type": type(without_plugin.payload.value[0]).__name__,
    "id unchanged": without_plugin.id == ext.id,
    "registered again": EXT_SCHEMA in bopp.extensions.get_extensions(),
}

{'warning': "No extension registered for schema 'io.github.johentsch.score_note:v1'. Skipping validation.",
 'row type': 'dict',
 'id unchanged': True,
 'registered again': True}

**Validation**
- With the plugin registered, bad rows are rejected when the annotation is created: an out-of-range `midi` value and an unknown field.

In [6]:
def creation_error(row):
    """The validation error raised when creating an ext annotation from ``row``, if any."""
    try:
        bopp.create(media_id="demo:bad", payload_kind="ext", ext_schema=EXT_SCHEMA, value=[row])
    except msgspec.ValidationError as err:
        return str(err)
    return None


{
    "midi out of range": creation_error({"midi": 128, "tpc": 0}),
    "unknown field": creation_error({"midi": 60, "tpc": 0, "unknown": 1}),
}

{'midi out of range': 'Expected `int` <= 127 - at `$[0].midi`',
 'unknown field': 'Object contains unknown field `unknown` - at `$[0]`'}

**Round trip**
- `from_ext` turns the rows back into columns and reproduces the original columnar id.

In [7]:
from_ext(to_ext(ann)).id == ann.id

True

**Timing**
- MsgPack decode time for the columnar form, the `ext` form without row validation, and the `ext` form with it.

In [8]:
def best_ms(stmt):
    """Best of 5 repeats of 20 calls, per call, in milliseconds."""
    return min(timeit.repeat(stmt, number=20, repeat=5)) / 20 * 1000


pd.Series(
    {
        "columnar": best_ms(lambda: load_bopp_msgpack(WORK / "columnar.msgpack")),
        "ext, first pass only": best_ms(lambda: load_bopp_msgpack(WORK / "ext.msgpack", resolve_ext=False)),
        "ext, both passes": best_ms(lambda: load_bopp_msgpack(WORK / "ext.msgpack")),
    },
    name="msgpack decode (ms)",
).round(3)

columnar                1.673
ext, first pass only    2.856
ext, both passes        2.742
Name: msgpack decode (ms), dtype: float64

The `ext` form has the same id with the plugin resolved, with rows left as dicts, and after a JSON or MsgPack round trip, because the rows serialize to the same bytes either way. Row validation is the second pass: it runs only when the plugin is registered, so a file stays readable without it and its id still checks out. The second pass (row validation) adds no measurable time here: the `ext` timings with and without it are within run-to-run noise. The `ext` file is larger and slower to decode than the columnar one, so the extra cost comes from the row-oriented first pass, not from validation. Two caveats from the plugin README remain: CSV round trips of `ext` payloads are not id-stable upstream yet, and a column that is entirely `None` does not survive `to_ext`.